In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

import seaborn as sns
from linearmodels.panel import PanelOLS

import statsmodels.api as sm


## We need to link to the datasets. These are too large to put on github, perhaps we should set up a dvc?
# !! UPDATE PATHS AS NEEDED  !!
# Matti here, saving his paths yeehaw
# C:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\data\BACI_sets    (or acled_sets, or gravity_sets)

BACI_folder_path_init = r"C:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\data\BACI_sets"
BACI_folder_path = Path(BACI_folder_path_init).as_posix()

ACLED_folder_path_init = r"C:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\data\acled_sets"
ACLED_folder_path = Path(ACLED_folder_path_init).as_posix()

Gravity_folder_path_init = r"C:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\data\gravity_sets"
Gravity_folder_path = Path(Gravity_folder_path_init).as_posix()

## Combining all of ACLED Africa, Gravity & Refugee

In [ ]:
# New link - We should combine all ACLED data from Africa with the Gravity dataset

## THIS IS DEPRECATED - IGNORE unless your name is Zen Rehda


#acled_af = f"{ACLED_folder_path}/africa_acled.csv"
#gravity = f"{Gravity_folder_path}/Gravity_V202211.csv"

#df_a = pd.read_csv(acled_af)
#df_g = pd.read_csv(gravity)


### 01. Prepare ACLED Africa

This run was done locally on Mattis Multimachine (Therefore we re-link the two folders above)
We need to clean up the ACLED data and only keep relevant data.

In [15]:
acled_af = f"{ACLED_folder_path}/ACLEDAfricaData_1997_2026-02-02.csv"
gravity = f"{Gravity_folder_path}/Gravity_V202211.csv"

df_a = pd.read_csv(acled_af)
df_g = pd.read_csv(gravity)

# We can filter to the relevant columns we want - More can be added here, but update dummy code below if need be
#   inter1 is the perpetrator, inter2 is the target
df_a_filter = df_a[["country", "year", "disorder_type", "event_type", "inter1", "inter2", "fatalities"]
].copy()

# We save a list of all unique possible values for the columns for future use and reference.
country_list = df_a_filter["country"].sort_values().unique()

disorder_types = df_a_filter["disorder_type"].unique()
event_types = df_a_filter["event_type"].unique()
attack_groups = df_a_filter["inter1"].unique()
target_groups = df_a_filter["inter2"].unique()

# Now we need to link the countries to the tags in Gravity, so the countries can be linked between the two datasets
df_a_filter.tail(5)

C:\Users\mhm25\AppData\Local\Temp\ipykernel_16080\4148789484.py:5: DtypeWarning: Columns (0: empire) have mixed types. Specify dtype option on import or set low_memory=False.
  df_g = pd.read_csv(gravity)


,country,year,disorder_type,event_type,inter1,inter2,fatalities
418055,Niger,2025,Political violence,Violence against civilians,Political militia,Civilians,0
418056,Cameroon,2025,Strategic developments,Strategic developments,Political militia,Civilians,0
418057,Cameroon,2025,Political violence,Violence against civilians,Rebel group,Civilians,0
418058,Cameroon,2025,Political violence,Violence against civilians,Rebel group,Civilians,0
418059,Cameroon,2025,Political violence,Violence against civilians,Rebel group,Civilians,0


In [ ]:
df_a_filter

In [16]:
df = df_a_filter.copy()

# We create dummy values for each type of disorder, event, attackers and target
dummies = pd.get_dummies(
    df[['disorder_type', 'event_type', 'inter1', 'inter2']],
    prefix=['disorder', 'event', 'perpetrator', 'target']
)

# We add the numeric columns back to the dummy dataset
dummies['fatalities'] = df['fatalities']
dummies['country'] = df['country']
dummies['year'] = df['year']

# Now we can group by country and year, and sum over the dummy categories. Perfect!
acled_result = dummies.groupby(['country', 'year']).sum().reset_index()

## We want to map the countries from ACLED onto the iso names from the Gravity dataset. I made this stupid dictionary BY HAND, please respect that.
country_iso_dict = {
    "Algeria": "DZA", "Angola": "AGO", 'Benin': "BEN", 'Botswana': "BWA", 'Burkina Faso': "BFA", "Burundi": "BDI", 'Cameroon': "CMR", 'Cape Verde': "CPV",
    "Central African Republic": "CAF", 'Chad': "TCD", 'Comoros': "COM", 'Democratic Republic of Congo': "COD", 'Djibouti': "DJI", 'Egypt': "EGY",
    'Equatorial Guinea': "GNQ", 'Eritrea': "ERI", 'Ethiopia': "ETH", 'Gabon': "GAB",'Gambia': "GMB", 'Ghana': "GHA", 'Guinea': "GIN", 'Guinea-Bissau': "GNB", 
    'Ivory Coast': "CIV", 'Kenya': "KEN", 'Lesotho': "LSO", 'Liberia': "LBR", 'Libya': "LBY", 'Madagascar': "MDG", 'Malawi': "MWI",'Mali': "MLI", 
    'Mauritania': "MRT", 'Mauritius': "MUS", 'Mayotte': "MYT", 'Morocco': "MAR",'Mozambique': "MOZ", 'Namibia': "NAM", 'Niger': "NER", 'Nigeria': "NGA", 
    'Republic of Congo': "COG", 'Reunion': "REU", 'Rwanda': "RWA", 'Saint Helena, Ascension and Tristan da Cunha': "SHN", 'Sao Tome and Principe': "STP", 
    'Senegal': "SEN", 'Seychelles': "SYC", 'Sierra Leone': "SLE", 'Somalia': "SOM", 'South Africa': "ZAF", 'South Sudan': "SSD", 'Sudan': "SDN", 
    'Tanzania': "TZA", 'Togo': "TGO", 'Tunisia': "TUN", 'Uganda': "UGA", 'Zambia': "ZMB", 'Zimbabwe': "ZWE", 'eSwatini': "SWZ"
}

# We update the dataframe to have a new column for the iso-tags.
acled_result["iso"] = acled_result["country"].map(country_iso_dict)

# ACLED is now ready for merging

### 02. Prepare Gravity

Gravity requires some extensive cleaning, as the dataset has a lot of superfluous data.

In [17]:
display(df_g.columns)

Index(['year', 'country_id_o', 'country_id_d', 'iso3_o', 'iso3_d', 'iso3num_o',
       'iso3num_d', 'country_exists_o', 'country_exists_d',
       'gmt_offset_2020_o', 'gmt_offset_2020_d', 'distw_harmonic',
       'distw_arithmetic', 'distw_harmonic_jh', 'distw_arithmetic_jh', 'dist',
       'main_city_source_o', 'main_city_source_d', 'distcap', 'contig',
       'diplo_disagreement', 'scaled_sci_2021', 'comlang_off', 'comlang_ethno',
       'comcol', 'col45', 'legal_old_o', 'legal_old_d', 'legal_new_o',
       'legal_new_d', 'comleg_pretrans', 'comleg_posttrans',
       'transition_legalchange', 'comrelig', 'heg_o', 'heg_d', 'col_dep_ever',
       'col_dep', 'col_dep_end_year', 'col_dep_end_conflict', 'empire',
       'sibling_ever', 'sibling', 'sever_year', 'sib_conflict', 'pop_o',
       'pop_d', 'gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 'pop_source_o',
       'pop_source_d', 'gdp_source_o', 'gdp_source_d', 'gdp_ppp_o',
       'gdp_ppp_d', 'gdpcap_ppp_o', 'gdpcap_ppp_d', 'pop_pwt_o',

In [18]:
# First, we define the parameters that sounded interesting to us after going over the documentation. Of these we will need to find the statistical significant ones 
target = ["iso3_o", "iso3_d", "country_exists_o", "country_exists_d", "distw_harmonic", "distw_arithmetic", "dist", "distcap", "diplo_disagreement", "scaled_sci_2021", "comlang_off", "comlang_ethno", "comleg_posttrans", "comrelig", "heg_o", "heg_d", "col_dep_ever", "col_dep", "col_dep_end_conflict", "sibling_ever", "sibling", "sever_year", "gdpcap_ppp", "wto", "eu", "fta_wto", "rta_type", "entry_tp", "tradeflow_comtrade_o", "tradeflow_comtrade_d", "tradeflow_baci", "manuf_tradeflow_baci", "tradeflow_imf_o", "tradeflow_imf_d"]


# Some parameters in the Gravity dataset differ from the documentation. These are below:
emp = []
for i in df_g.columns:
    if i in target:
        emp.append(i)

for i in target:
    if i not in emp:
        print(i)


gdpcap_ppp
wto
eu
entry_tp


In [19]:
## We add the missing parameters:
# gdpcap_ppp_o and d
# wto_o and d
# eu_o and d
# entry_tp_o and d

# Now we can start cleaning up the Gravity dataset to prepare for combining with the ACLED dataframe above
# REMOVED from high number of NaN: "scaled_sci_2021", "col_dep_end_conflict","rta_type", "tradeflow_comtrade_o", "tradeflow_comtrade_d","tradeflow_imf_o", "tradeflow_imf_d", "entry_tp_o", "entry_tp_d",

df_g_filter = df_g[["year", "iso3_o", "iso3_d", "country_exists_o", "country_exists_d", "distw_harmonic", "distw_arithmetic", "dist", "distcap", "diplo_disagreement",  "comlang_off", "comlang_ethno", "comleg_posttrans", "comrelig", "heg_o", "heg_d", "col_dep_ever", "col_dep",  "sibling_ever", "sibling", "sever_year", "gdpcap_ppp_o", "gdpcap_ppp_d", "wto_o", "wto_d", "eu_o", "eu_d","fta_wto",  "tradeflow_baci", "manuf_tradeflow_baci"]
]

display(df_g_filter.head())

,year,iso3_o,iso3_d,country_exists_o,country_exists_d,distw_harmonic,distw_arithmetic,dist,distcap,diplo_disagreement,...,sever_year,gdpcap_ppp_o,gdpcap_ppp_d,wto_o,wto_d,eu_o,eu_d,fta_wto,tradeflow_baci,manuf_tradeflow_baci
0,1948,ABW,ABW,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,ABW,ABW,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,ABW,ABW,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,ABW,ABW,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,ABW,ABW,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
country_iso_dict.values()

dict_values(['DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'CIV', 'KEN', 'LSO', 'LBR', 'LBY', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MYT', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 'COG', 'REU', 'RWA', 'SHN', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 'ZMB', 'ZWE', 'SWZ'])

In [21]:
''' We can now apply a lot of clean-up filters: 

- remove any rows for years before 1997
- remove rows where any of the two countries do not exist
- remove rows where origin and destination country is identical (these are silly)
- remove rows where neither origin or destination is in Africa (this we can change later if we go beyond Africa)
'''

# We can now apply a lot of clean-up filters: 
# 
# Filter if any of the countries do not exist AND remove any data from before the year 1997.

df_g_filter = df_g_filter[
    (df_g["year"] >= 1997) &
    (df_g["country_exists_o"] == 1) &
    (df_g["country_exists_d"] == 1) &
    (df_g["iso3_o"] != df_g["iso3_d"]) &
    (
        (df_g["iso3_o"].isin(country_iso_dict.values())) |
        (df_g["iso3_d"].isin(country_iso_dict.values()))
    )
]

# This gives us ~1420 origin rows per country 

# Gravity is now ready for merging.

In [22]:
df_g_filter

,year,iso3_o,iso3_d,country_exists_o,country_exists_d,distw_harmonic,distw_arithmetic,dist,distcap,diplo_disagreement,...,sever_year,gdpcap_ppp_o,gdpcap_ppp_d,wto_o,wto_d,eu_o,eu_d,fta_wto,tradeflow_baci,manuf_tradeflow_baci
197,1997,ABW,AGO,1,1,9590.0,9593.0,9505.0,9505.0,NaN,...,NaN,NaN,2.542,0.0,1.0,0.0,0.0,0.0,NaN,NaN
198,1998,ABW,AGO,1,1,9590.0,9593.0,9505.0,9505.0,NaN,...,NaN,NaN,2.671,0.0,1.0,0.0,0.0,0.0,NaN,NaN
199,1999,ABW,AGO,1,1,9590.0,9593.0,9505.0,9505.0,NaN,...,NaN,NaN,2.721,0.0,1.0,0.0,0.0,0.0,NaN,NaN
200,2000,ABW,AGO,1,1,9584.0,9587.0,9505.0,9505.0,NaN,...,NaN,NaN,2.781,0.0,1.0,0.0,0.0,0.0,NaN,NaN
201,2001,ABW,AGO,1,1,9584.0,9587.0,9505.0,9505.0,NaN,...,NaN,NaN,2.870,0.0,1.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4699217,2017,ZWE,ZMB,1,1,484.0,511.0,399.0,399.0,1.266,...,1964.0,2.416,3.485,1.0,1.0,0.0,0.0,1.0,82443.668,57765.023
4699218,2018,ZWE,ZMB,1,1,484.0,511.0,399.0,399.0,0.791,...,1964.0,2.557,3.606,1.0,1.0,0.0,0.0,1.0,78839.303,60417.984
4699219,2019,ZWE,ZMB,1,1,484.0,511.0,399.0,399.0,0.124,...,1964.0,2.408,3.617,1.0,1.0,0.0,0.0,1.0,77331.594,58542.078
4699220,2020,ZWE,ZMB,1,1,479.0,505.0,399.0,399.0,0.226,...,1964.0,2.251,3.457,1.0,1.0,0.0,0.0,1.0,71653.898,53614.934


### 03. Preparing the refugee data.

We have a small extra dataset for refugee data that can be added for extra fun!

In [ ]:
refugee_path_init = r"C:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\bachelor_2026\data\sit_sets\persons_of_concern.csv"
refugee_path = Path(refugee_path_init).as_posix()

# Now we can include the refugee data.
df_s = pd.read_csv(refugee_path)


In [ ]:
# Let's rename the columns to fit the conventions of Gravity
df_s = df_s.rename(columns={
    "Year": "year",
    "Country of Asylum": "ref_country_d",
    "Country of Origin": "ref_country_o",
    "Country of Asylum ISO": "ref_iso_d",
    "Country of Origin ISO": "ref_iso_o"    
})

In [ ]:
df_s.columns


In [ ]:
'''
WIP for the data filtering on the refugee data. We want to remove:

- We remove all rows from before 1997
- We remove all refugees whose origin is "Unknown" (we want the country of origin to be known for training)
'''

df_s_filter = df_s[
    (df_s["year"] >= 1997) &                            
    (df_s["ref_country_o"] != "Unknown ")
]



df_s_filter = df_s_filter[
    (df_s["year"] >= 1997) &
    (df_s["ref_iso_o"] != df_s["ref_iso_d"]) &
    (
        (df_s["ref_iso_o"].isin(country_iso_dict.values())) |
        (df_s["ref_iso_d"].isin(country_iso_dict.values()))
    )
]

df_s_filter

In [ ]:
df_merged_full_situ = pd.merge(
    df_s_filter, 
    df_g_filter, 
    left_on=['ref_iso_o', 'year'], 
    right_on=['iso3_o', 'year'], 
    how='inner'
)
print(f"Situation filter shape: {df_s_filter.shape}")
print(f"Gravity filter shape: {df_g_filter.shape}")
print(f"Merged master shape: {df_merged_full_situ.shape}")



In [ ]:
df_merged_full_situ

### 04. Merging the two datasets - ACLED and Gravity

Time to put our differences aside and get these two datasets into one grand dataset! 

In [23]:
df_merged_full = pd.merge(
    acled_result, 
    df_g_filter, 
    left_on=['iso', 'year'], 
    right_on=['iso3_o', 'year'], 
    how='inner'
)
print(f"Raw ACLED shape: {df_a.shape}")
print(f"Gravity filter shape: {df_g_filter.shape}")
print(f"Merged master shape: {df_merged_full.shape}")



Raw ACLED shape: (418060, 31)
Gravity filter shape: (579204, 30)
Merged master shape: (283311, 59)


In [24]:
# We have some extra / superfluous columns. Let's drop them 
#
# iso - We have multiple of these
# country_exists - We filter for these earlier, no longer useful.

df_merged_full = df_merged_full.drop(columns=['iso', 'country_exists_o', 'country_exists_d'])

In [ ]:
df_merged_full

In [25]:
# To check the NaN counts in our data   (See notes_for_work.md 04/03 entry to find reasons for removing columns)
na_counts = df_merged_full.isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)
na_counts


sever_year              234470
manuf_tradeflow_baci    147439
tradeflow_baci          147439
diplo_disagreement       69965
gdpcap_ppp_d             62533
comrelig                 46285
gdpcap_ppp_o             19569
comlang_off              11991
comlang_ethno            11991
comleg_posttrans            16
dtype: int64

### 05. Looking at specific conflicts 


#### The Gambia 
In 2016, a constitutional crisis erupted in the Gambia, escalating into a full-blown military intervention by the other nations of ECOWAS, who have since had troops stationed in the country.


#### Ghana
Small regional conflicts and rebellions broke out in Central and Eastern Ghana in 2019. 


#### Kenya
In the years following 2005, a number of conflicts erupted in Kenya, including a political crisis and a terrorist attack in 2013.


In [ ]:
# We can set up list for the largest economies. This is to check integration in the global market.
largest_economies = ["RUS", "USA", "CHN", "DEU", "ITA", "JPN", "IND", "GBR", "FRA", "CAN"]


In [ ]:
## DataFrame for Kenya
# We can look at their neighbors. 
kenya_neighbors = ["SSD", "ETH", "SOM", "TZA", "UGA"]

# And also members of COMESA (Common Market for Eastern and Southern Africa)
kenya_comesa = ["DJI", "ERI", "ETH", "SOM", "EGY", "LBY", "SDN", "TUN", "COM", "MDG", "MUS", "SYC", "BDI", "MWI", "RWA", "UGA", "SWZ", "ZMB", "ZWE", "COD"] 


kenya_df_neighbors = df_merged_full[
    (df_merged_full["country"] == "Kenya") &
    (df_merged_full["iso3_d"].isin(kenya_neighbors))
]

kenya_df_comesa = df_merged_full[
    (df_merged_full["country"] == "Kenya") &
    (df_merged_full["iso3_d"].isin(kenya_comesa))
]

kenya_df_largest = df_merged_full[
    (df_merged_full["country"] == "Kenya") &
    (df_merged_full["iso3_d"].isin(largest_economies))
]

In [54]:
## DataFrame for Gambia
# We want to include all members of the ECOWAS union
# AS WELL as the nations that were excluded since 2021. (These include Mali, Guinea, Burkina Faso, )
gambia_neighbors_ecowas = ["SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
gambia_ecowas_previous_members = ["MLI", "BFA", "NER"]


gambia_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Gambia") &
    (df_merged_full["iso3_d"].isin(gambia_neighbors_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))]

gambia_df_largest = df_merged_full[
    (df_merged_full["country"] == "Gambia") &
    (df_merged_full["iso3_d"].isin(largest_economies) )]




# We can prepare dataframes for all other ecowas member states: 
senegal_ecowas = ["GMB", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
senegal_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Senegal") & 
    (df_merged_full["iso3_d"].isin(senegal_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

benin_ecowas = ["GMB", "SEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
benin_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Benin") &
    (df_merged_full["iso3_d"].isin(benin_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

capeV_ecowas = ["GMB", "SEN", "BEN", "GHA", "GIN", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
capeV_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Cape Verde") &
    (df_merged_full["iso3_d"].isin(capeV_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

ghana_ecowas = ["GMB", "SEN", "BEN", "CPV", "GIN", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
ghana_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Ghana") &
    (df_merged_full["iso3_d"].isin(ghana_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

guinea_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GNB", "CIV", "LBR", "NGA", "SLE", "TGO"]
guinea_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Guinea") &
    (df_merged_full["iso3_d"].isin(guinea_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

guineaB_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "CIV", "LBR", "NGA", "SLE", "TGO"]
guineaB_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Guinea-Bissau") &
    (df_merged_full["iso3_d"].isin(guineaB_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

coteI_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "LBR", "NGA", "SLE", "TGO"]
coteI_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Ivory Coast") &
    (df_merged_full["iso3_d"].isin(coteI_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

liberia_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "NGA", "SLE", "TGO"]
liberia_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Liberia") &
    (df_merged_full["iso3_d"].isin(liberia_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

nigeria_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LIB", "SLE", "TGO"]
nigeria_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Nigeria") &
    (df_merged_full["iso3_d"].isin(nigeria_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

sierraL_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LIB", "NGA", "TGO"]
sierraL_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Sierra Leone") &
    (df_merged_full["iso3_d"].isin(sierraL_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

togo_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LIB", "NGA", "SLE"]
togo_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Togo") &
    (df_merged_full["iso3_d"].isin(togo_ecowas)  | df_merged_full["iso3_d"].isin(gambia_ecowas_previous_members))
]

# And now for the recently excluded members:
full_ecowas = ["GMB", "SEN", "BEN", "CPV", "GHA", "GIN", "GNB", "CIV", "LIB", "NGA", "SLE", "TGO"]

mali_ecowas_prev = ["BFA", "NER"]
mali_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Mali") &
    (df_merged_full["iso3_d"].isin(full_ecowas)  | df_merged_full["iso3_d"].isin(mali_ecowas_prev))
]

burkinaF_ecowas_prev = ["MLI", "NER"]
burkinaF_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Burkina Faso") &
    (df_merged_full["iso3_d"].isin(full_ecowas)  | df_merged_full["iso3_d"].isin(burkinaF_ecowas_prev))
]

niger_ecowas_prev = ["MLI",  "BFA"]
niger_df_ecowas = df_merged_full[
    (df_merged_full["country"] == "Niger") &
    (df_merged_full["iso3_d"].isin(full_ecowas)  | df_merged_full["iso3_d"].isin(niger_ecowas_prev))
]

ecowas_df_dict = {"Senegal": senegal_df_ecowas,
                   "Benin": benin_df_ecowas,
                   "Cape Verde": capeV_df_ecowas,
                   "Ghana": ghana_df_ecowas,
                   "Guinea": guinea_df_ecowas,
                   "Guinea-Bissau": guineaB_df_ecowas,
                   "Ivory Coast": coteI_df_ecowas,
                   "Liberia": liberia_df_ecowas,
                   "Nigeria": nigeria_df_ecowas,
                   "Sierra Leone": sierraL_df_ecowas,
                   "Togo": togo_df_ecowas,

                   "Mali": mali_df_ecowas,
                   "Burkina Faso": burkinaF_df_ecowas,
                   "Niger": niger_df_ecowas
                   }

In [55]:
# We look at the case of the Gambia, our chosen conflict happening in 2016.

pre_years    = list(range(1997, 2016))  # 1997–2015
crisis_years = [2016, 2017]             # crisis span at annual resolution
post_years   = list(range(2018, 2022))  # 2018–2021

periods = {
    "normal": (1997, 2015),
    "crisis": (2016, 2017),
    "post_crisis": (2018, 2022)
}



def filter_period(df, start, end):
    '''
    Super simple function to define a period to filter after in a dataframe.
    '''
    return df[(df["year"] >= start) & (df["year"] <= end)]

In [ ]:
country_period_slices = {}

for country, df in ecowas_df_dict.items():
    country_period_slices[country] = {
        period_name: filter_period(df, *date_range)
            for period_name, date_range in periods.items()
    }

In [ ]:
# Here are examples how to access the slices

country_period_slices["Senegal"]["crisis"]
country_period_slices["Burkina Faso"]["normal"]
country_period_slices["Ghana"]["post_crisis"]


,country,year,disorder_Demonstrations,disorder_Political violence,disorder_Political violence; Demonstrations,disorder_Strategic developments,event_Battles,event_Explosions/Remote violence,event_Protests,event_Riots,...,sever_year,gdpcap_ppp_o,gdpcap_ppp_d,wto_o,wto_d,eu_o,eu_d,fta_wto,tradeflow_baci,manuf_tradeflow_baci
103456,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,3.236,1.0,1.0,0.0,0.0,1.0,36348.115,36218.313
103458,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,2.168,1.0,1.0,0.0,0.0,1.0,319802.328,251339.688
103479,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,5.154,1.0,1.0,0.0,0.0,1.0,99132.019,84290.555
103486,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,7.028,1.0,1.0,0.0,0.0,0.0,815.895,739.572
103517,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,2.559,1.0,1.0,0.0,0.0,1.0,6311.653,6294.741
103519,Ghana,2018,42,52,1,6,14,0,32,30,...,1957.0,5.442,2.210,1.0,1.0,0.0,0.0,1.0,2342.453,2342.205
103520,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,1.947,1.0,1.0,0.0,0.0,1.0,319.561,319.362
103556,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,1.581,1.0,1.0,0.0,0.0,1.0,15276.998,15207.069
103574,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,2.338,1.0,1.0,0.0,0.0,1.0,68903.665,64760.512
103590,Ghana,2018,42,52,1,6,14,0,32,30,...,NaN,5.442,1.229,1.0,1.0,0.0,0.0,1.0,70551.858,65163.414


In [ ]:
# Deprecated for now, but this might be useful for the above slices?

from sklearn.feature_selection import mutual_info_regression

df = gambia_df_largest.copy()
df = df.drop(columns=["diplo_disagreement", "iso3_o", "sever_year"])
#df = df.drop(columns=["country", "iso3_o", "iso3_d", "sever_year"])
df = df.dropna()


### PanelOLS setup for Gambia

We want to check normal years of growth and establish this normalcy, to compare with the crisis years

In [ ]:
gambia_df_ecowas

In [ ]:
print(event_types)
print(disorder_types)

In [ ]:
# Create panel index
df_panel = gambia_df_ecowas.set_index(["iso3_d", "year"]).sort_index()

# Some of the column names are slightly wacky. Maybe we can just straight up rename them? 
# df_panel = df_panel.rename(columns=lambda c: c.replace(" ", "_"))


# There may be an issue with heteroskedasticity in the data, where the variance in the error terms across all levels of the independent variables are not constant. 
# "It violates the standard OLS assumption of constant variance"
# We can set the trade to be logarithmic, to handle outliers and this issue.
df_panel["log_trade"] = np.log1p(df_panel["tradeflow_baci"])



model_parameters = [
    "log_trade",
    "event_Protests",
    "event_Battles",
    "event_Violence against civilians",
    "event_Strategic developments",
    "event_Explosions/Remote violence",
    "event_Riots",
    "disorder_Political violence",
    "disorder_Demonstrations",
    "disorder_Strategic developments",
    "disorder_Political violence; Demonstrations"
]
dfm = df_panel.dropna(subset=model_parameters).copy()


# Found this smart sanity check! Let's make sure there are no duplicates in the data.
dups = dfm.reset_index().duplicated(subset=["iso3_d", "year"]).sum()
if dups > 0:
    print("Warning: duplicate iso3_d-year rows:", dups)



model = PanelOLS.from_formula(
    'log_trade ~  Q("event_Protests") + Q("event_Battles") + Q("event_Violence against civilians") + Q("event_Strategic developments") + Q("event_Explosions/Remote violence") + Q("event_Riots") + Q("disorder_Political violence") + Q("disorder_Demonstrations") + Q("disorder_Strategic developments") + Q("disorder_Political violence; Demonstrations") + EntityEffects',
    data=dfm
)

# Cluster SEs at the country (entity) level – common in panel trade/conflict work
res = model.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res)



# Descriptive: years with any incidents by partner (no Q() in pandas)
cols = [
    "event_Protests",
    "event_Battles",
    "event_Violence against civilians",
    "event_strategic developments",
    "event_Explosions/Remote violence",
    "event_Riots",
    "disorder_Political violence",
    "disorder_Demonstrations",
    "disorder_Strategic developments",
    "disorder_Political violence; Demonstrations"
]
existing = [c for c in cols if c in df_panel.columns]
if existing:
    years_with_incidents = (df_panel[existing] > 0).groupby(level="iso3_d").sum()
    print(years_with_incidents.sort_index())



#df_panel.groupby("iso3_d")[["disorder_Demonstrations" + "event_Protests" + "event_Battles" + "disorder_Political violence"]].nunique()

In [ ]:

# df_panel indexed by (iso3_d, year), as you have
y = df_panel["log_trade"]

# List of all ACLED candidates you want to screen
candidates = [
    "event_Protests",
    "event_Battles",
    "event_Violence against civilians",
    "event_Strategic developments",
    "event_Explosions/Remote violence",
    "event_Riots",
    "disorder_Political violence",
    "disorder_Demonstrations",
    "disorder_Strategic developments",
    # Avoid composite that is an exact sum of others here
]

# 1) Residualize y on entity FE
D_entity = pd.get_dummies(df_panel.index.get_level_values("iso3_d"), drop_first=True)
y_hat = sm.OLS(y, sm.add_constant(D_entity, has_constant='add')).fit().fittedvalues
y_res = y - y_hat

screen_results = []
for var in candidates:
    if var not in df_panel.columns:
        continue
    x = df_panel[var]
    # 2) Residualize x on entity FE
    x_hat = sm.OLS(x, sm.add_constant(D_entity, has_constant='add')).fit().fittedvalues
    x_res = x - x_hat

    # Optional: transform intensity to stabilize variance
    # x_res = np.log1p(np.maximum(x_res + x_res.min(), 0))  # use with caution; usually transform before residualizing

    # 3) OLS of residualized y on residualized x
    m = sm.OLS(y_res, sm.add_constant(x_res, has_constant='add')).fit(cov_type="HC1")
    screen_results.append({
        "variable": var,
        "beta": m.params.get(x_res.name, np.nan),
        "pval": m.pvalues.get(x_res.name, np.nan),
        "t": m.tvalues.get(x_res.name, np.nan),
        "n": int(m.nobs)
    })

screen_df = pd.DataFrame(screen_results).sort_values("pval")
print(screen_df)

In [ ]:
df_pre = df[
    df["year"].isin(pre_years)
]

df_crisis = df[
    df["year"].isin(crisis_years)
]

df_post = df[
    df["year"].isin(post_years)
]

In [ ]:
sns.lineplot(
    data=df,
    x="year",
    y="tradeflow_baci",
    hue="iso3_d"
)


In [ ]:

sns.lineplot(
    data=df_pre,
    x="year",
    y="tradeflow_baci",
    hue="iso3_d"
)


In [ ]:
df_pre.groupby(["year", "iso3_d"])["tradeflow_baci"].sum().reset_index()

In [ ]:
df_pre

In [ ]:
# We want to find a relationship between the conflicts and the different features in the dataframe.
# This sadly only shows that gdp and conflicts are symmetrically informative. 
# Which just goes with empirical data, but doesn't tell us anything new. 

# We find all the features from Gravity to use in a for loop
gravity_features = df.columns[27:]

mi_results = {}

string_drop = df.columns[27:]
X = df.drop(columns=string_drop)


for i in range(len(gravity_features)):
    #X = df.drop(columns=[f"{gravity_features[i]}"])
    y = df[f"{gravity_features[i]}"]

    mi_scores = mutual_info_regression(X, y, random_state=42)
    mi_scores = pd.Series(mi_scores, index=X.columns)

    mi_results[gravity_features[i]] = mi_scores.sort_values(ascending=False)


mutual_info_df = pd.DataFrame(mi_results)

mutual_info_df = mutual_info_df.drop(columns={"distw_harmonic", "distw_arithmetic", "dist", "distcap", "heg_o", "heg_d", "eu_o", "eu_d", "wto_o", "wto_d", "fta_wto", "comrelig", "comlang_off", "comlang_ethno", "comleg_posttrans"})

mutual_info_df

In [ ]:
# Need a few sklearn functions imported 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold

In [ ]:
# Set default dataframe and ensure year is the right type (int)
df = gambia_df_ecowas.copy()
df['year'] = df['year'].astype(int)

# Target column, tradeflow at the moment, but can be set to anything
ycol = 'tradeflow_baci'

# Identify ACLED columns
event_cols    = [c for c in df.columns if c.startswith('event_')]
disorder_cols = [c for c in df.columns if c.startswith('disorder_')]
other_counts  = [c for c in ['fatalities'] if c in df.columns]
acled_cols    = event_cols + disorder_cols + other_counts

# Totals and shares (optional, for later specs)
df['events_total']   = df[event_cols].sum(axis=1, skipna=True) if event_cols else 0
df['disorder_total'] = df[disorder_cols].sum(axis=1, skipna=True) if disorder_cols else 0
for c in event_cols:
    df[f'share_{c}'] = df[c] / df['events_total'].replace(0, np.nan)
for c in disorder_cols:
    df[f'share_{c}'] = df[c] / df['disorder_total'].replace(0, np.nan)

# Mark Senegal neighbor
df['neighbor'] = (df['iso3_d'] == 'SEN').astype(int)

# Windows (you can tweak)
pre_years    = list(range(2012, 2016))  # 2012–2015
crisis_years = [2016, 2017]             # crisis span at annual resolution
post_years   = list(range(2018, 2022))  # 2018–2021

In [ ]:
df

In [ ]:
# Aggregate trade & ACLED to ECOWAS total per year
year_agg = (df.groupby('year', as_index=False)
              [[ycol] + acled_cols].sum()
              .sort_values('year'))

# Δlog and % changes
year_agg['trade_log']      = np.log1p(year_agg[ycol].fillna(0))
year_agg['d_trade_log']    = year_agg['trade_log'].diff()
year_agg['pct_change_baci']= year_agg[ycol].pct_change() * 100

# Also difference ACLED (optional)
for c in acled_cols:
    year_agg[f'd_{c}'] = year_agg[c].diff()

def corr_table(df_in, y_var, x_vars):
    tmp = df_in[[y_var] + x_vars].dropna()
    rows = []
    for x in x_vars:
        pear = tmp[[y_var, x]].corr(method='pearson').iloc[0,1]
        spear = tmp[[y_var, x]].corr(method='spearman').iloc[0,1]
        rows.append({'var': x, 'pearson': pear, 'spearman': spear, 'N': len(tmp)})
    return pd.DataFrame(rows).sort_values('pearson', ascending=False)

# Correlate Δlog(trade) with ACLED levels and with ΔACLED
corr_levels = corr_table(year_agg, 'd_trade_log', acled_cols)
corr_diffs  = corr_table(year_agg, 'd_trade_log', [f'd_{c}' for c in acled_cols])

In [ ]:
corr_levels

In [ ]:
# Build Δlog per partner-year
panel = df.sort_values(['iso3_d','year']).copy()
panel['ln_trade']   = np.log1p(panel[ycol].fillna(0))
panel['d_ln_trade'] = panel.groupby('iso3_d')['ln_trade'].diff()

# Drop first-diff NA
panel = panel.dropna(subset=['d_ln_trade'])

# Design matrix:
# d_ln_trade_{p,t} = alpha_p + beta * ACLED_t + gamma * (ACLED_t × neighbor_p) + e
X_acled = panel[acled_cols].copy().fillna(0)

# Standardize ACLED predictors so coefficients are comparable “weights”
sc_acled = StandardScaler()
X_acled_std = pd.DataFrame(sc_acled.fit_transform(X_acled), columns=acled_cols, index=X_acled.index)

# Interaction with neighbor
X_nei = X_acled_std.mul(panel['neighbor'].values.reshape(-1,1))
X_nei.columns = [f'neiX_{c}' for c in X_acled_std.columns]

# Partner fixed effects via dummies (drop_first to avoid full collinearity)
D_partner = pd.get_dummies(panel['iso3_d'], drop_first=True)
X_design = pd.concat([D_partner, X_acled_std, X_nei], axis=1)

y = panel['d_ln_trade'].values

# Use Ridge for stability (small N per partner, many predictors)
# CV folds: keep between 3 and 10
years_available = panel['year'].nunique()
K = max(3, min(10, years_available))
ridge = RidgeCV(cv=K).fit(X_design.values, y)

coef_series = pd.Series(ridge.coef_, index=X_design.columns)
# Extract standardized weights for ACLED terms
weights_main = coef_series[acled_cols].sort_values(ascending=False)
weights_neix = coef_series[[c for c in coef_series.index if c.startswith('neiX_')]].sort_values(ascending=False)

# Optional: plain OLS (LinearRegression) if you want unpenalized estimates
ols = LinearRegression().fit(X_design.values, y)
coef_ols = pd.Series(ols.coef_, index=X_design.columns)
weights_main_ols = coef_ols[acled_cols].sort_values(ascending=False)
weights_neix_ols = coef_ols[[c for c in coef_ols.index if c.startswith('neiX_')]].sort_values(ascending=False)

In [ ]:
# Year-level Δlog target
Y = year_agg[['year','d_trade_log']].dropna().copy()
Xy = year_agg.loc[Y.index, acled_cols].fillna(0)

# Standardize predictors
sc_year = StandardScaler()
Xy_std = sc_year.fit_transform(Xy.values)
y_year = Y['d_trade_log'].values

# LASSO with cross-validation (works well with small N to select variables)
K = max(3, min(10, len(Y)))
lasso = LassoCV(cv=K, random_state=42).fit(Xy_std, y_year)
lasso_coefs = pd.Series(lasso.coef_, index=Xy.columns).sort_values(ascending=False)
lasso_keep  = lasso_coefs[lasso_coefs != 0]

# Permutation importance (model-agnostic ranking)
perm = permutation_importance(lasso, Xy_std, y_year, n_repeats=200, random_state=42)
perm_imp = (pd.DataFrame({
    'feature': Xy.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False))

In [ ]:

print("\n=== Year-level correlations (levels) with Δlog(tradeflow_baci) ===")
print(corr_levels.head(12).to_string(index=False))

print("\n=== Year-level correlations (diffs) with Δlog(tradeflow_baci) ===")
print(corr_diffs.head(12).to_string(index=False))

print("\n=== Panel standardized weights (RidgeCV): ACLED main effects ===")
print(weights_main.round(3).head(15).to_string())

print("\n=== Panel standardized weights (RidgeCV): ACLED x Neighbor (Senegal) ===")
print(weights_neix.round(3).head(15).to_string())

print("\n=== LASSO (year-level): non-zero standardized coefficients ===")
print(lasso_keep.round(4).to_frame('lasso_coef_std').head(20).to_string())

print("\n=== Permutation importance (year-level) ===")
print(perm_imp.head(15).to_string(index=False))
